In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName('Testing pyspar example').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/27 11:08:14 WARN Utils: Your hostname, Usmanovs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.25.235.138 instead (on interface en0)
26/10/27 11:08:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/27 11:08:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/27 11:08:16 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
sample_data = [{"name": "John    D.", "age": 30},
  {"name": "Alice   G.", "age": 25},
  {"name": "Bob  T.", "age": 35},
  {"name": "Eve   A.", "age": 28}]

df = spark.createDataFrame(sample_data)

In [3]:
df.show()

+---+----------+
|age|      name|
+---+----------+
| 30|John    D.|
| 25|Alice   G.|
| 35|   Bob  T.|
| 28|  Eve   A.|
+---+----------+



In [4]:
from pyspark.sql.functions import col,regexp_replace

# this function remove additional spaces in name
def remove_extra_spaces(df,col_name):
    # removeing extrac spaces from the specified col 
    df_transformed = df.withColumn(col_name,regexp_replace(col(col_name), "\\s+"," "))
    return df_transformed

transformed_df = remove_extra_spaces(df,"name")
transformed_df.show()

+---+--------+
|age|    name|
+---+--------+
| 30| John D.|
| 25|Alice G.|
| 35|  Bob T.|
| 28|  Eve A.|
+---+--------+



In [5]:
import pyspark.testing
from pyspark.testing.utils import assertDataFrameEqual

# Example 1
df1 = spark.createDataFrame(data=[("1", 1000), ("2", 3000)], schema=["id", "amount"])
df2 = spark.createDataFrame(data=[("1", 1000), ("2", 3000)], schema=["id", "amount"])
assertDataFrameEqual(df1, df2)  # pass, DataFrames are identical

In [6]:
# Example 2
df1 = spark.createDataFrame(data=[("1", 0.1), ("2", 3.23)], schema=["id", "amount"])
df2 = spark.createDataFrame(data=[("1", 0.109), ("2", 3.23)], schema=["id", "amount"])
assertDataFrameEqual(df1, df2, rtol=1e-1)  # pass, DataFrames are approx equal by rtol

In [7]:
import unittest

class PySparkTestCase(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.spark = SparkSession.buidler.appName('testi')

In [8]:
import pytest

In [9]:
@pytest.fixture
def spark_fixture():
    spark = SparkSession.builder.appName('testing pyspar example').getOrCreate()
    yield spark
    

In [15]:
def test_single_space(spark_fixture):
    sample_data = [{"name": "John    D.", "age": 30},
                   {"name": "Alice   G.", "age": 25},
                   {"name": "Bob  T.", "age": 35},
                   {"name": "Eve   A.", "age": 28}]
    original_df = spark_fixture.createDataFrame(sample_data)
    transformed_df = remove_extra_spaces(original_df,'name')


    expected_data = [{"name": "John D.", "age": 30},
    {"name": "Alice      G.", "age": 25},
    {"name": "Bob T.", "age": 35},
    {"name": "Eve A.", "age": 28}]

    expected_df = spark_fixture.createDataFrame(expected_data)
    assertDataFrameEqual(transformed_df,expected_df)

In [16]:
test_single_space(spark)

PySparkAssertionError: [DIFFERENT_ROWS] Results do not match: ( 25.00000 % )
*** actual ***
[31m! Row(age=25, name='Alice G.')[0m
  Row(age=28, name='Eve A.')
  Row(age=30, name='John D.')
  Row(age=35, name='Bob T.')


*** expected ***
[31m! Row(age=25, name='Alice      G.')[0m
  Row(age=28, name='Eve A.')
  Row(age=30, name='John D.')
  Row(age=35, name='Bob T.')